In [ ]:
# Upload CSV
from google.colab import files
uploaded = files.upload()

import io
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

filename = list(uploaded.keys())[0]

# ✅ Fix: handle commas inside message text
df = pd.read_csv(io.BytesIO(uploaded[filename]), quoting=0, on_bad_lines='skip')

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

df['label'] = df['label'].str.strip().map({'ham': 0, 'spam': 1})
df = df.dropna(subset=['label', 'message'])  # drop any broken rows

X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

def predict_message(text):
    return "Spam" if model.predict(vectorizer.transform([text]))[0] == 1 else "Ham"

print(predict_message("You won a free prize"))
print(predict_message("Let's meet tomorrow"))